In [60]:
from yfinance import download
import pandas as pd
import numpy as np
from scipy.stats import linregress
from plotly.subplots import make_subplots
import math

import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [61]:
tickers = [
    "ABEV3.SA", "ALOS3.SA", "AXIA6.SA", "B3SA3.SA", "BBDC3.SA", "BBDC4.SA",
    "BPAC11.SA", "BRAP4.SA", "BRAV3.SA", "CPFE3.SA", "CPLE3.SA", "CURY3.SA",
    "EGIE3.SA", "EMBJ3.SA", "ENEV3.SA", "ENGI11.SA", "EQTL3.SA", "FLRY3.SA",
    "GGBR4.SA", "GOAU4.SA", "IGTI11.SA", "IRBR3.SA", "ISAE4.SA", "ITSA4.SA",
    "ITUB4.SA", "KLBN11.SA", "MOTV3.SA", "MRVE3.SA", "MULT3.SA", "PRIO3.SA",
    "PSSA3.SA", "RADL3.SA", "RENT3.SA", "SANB11.SA", "SBSP3.SA", "SLCE3.SA",
    "SUZB3.SA", "TAEE11.SA", "TIMS3.SA", "UGPA3.SA", "USIM5.SA", "VALE3.SA",
    "VBBR3.SA", "VIVT3.SA", "WEGE3.SA",

    '^BVSP'
]

param = {
    'interval': '1d',
    'period': '10y',
}

df_core = download(tickers, **param, auto_adjust=False)
df_core = df_core.loc[:, df_core.isna().sum() <= 200] # remover colunas com mais de 1000 valores faltantes
df_core = df_core.dropna()
df_core = df_core['Adj Close']
df_ret = df_core.pct_change().dropna()

[*********************100%***********************]  46 of 46 completed


# **Preço**

## Interpretação da regressão linear (preço vs tempo)

**Slope**
- Variação média diária do preço.
- `slope = 2,0`: preço sobe, em média, 2 unidades por dia.
- `slope = -0,5`: preço cai, em média, 0,5 unidades por dia.
- Indica direção e velocidade da tendência.

**Intercept**
- Valor estimado no ponto inicial da série temporal.
- Serve como ponto de partida da reta de regressão.
- Tem menor relevância para análise de tendência.

**R-value**
- Coeficiente de correlação entre tempo e preço.
- Vai de `-1` a `1`.
- `|r| próximo de 1`: tendência linear forte.
- `|r| próximo de 0`: tendência linear fraca ou inexistente.

**P-value**
- Significância estatística da tendência.
- `p < 0,05`: forte evidência de tendência consistente.
- `p >= 0,05`: tendência pode ser aleatória.

**Std err**
- Erro padrão do slope (incerteza da estimativa).
- Valor menor: estimativa mais precisa.
- Valor maior: estimativa menos confiável.

In [62]:
# regressão linear contra o tempo para cada ativo

resultados = {}

x = np.arange(len(df_core))  # eixo tempo

for ticker in df_core.columns:

    y = df_core[ticker]

    x_adj = x[-len(y):]

    slope, intercept, r_value, p_value, std_err = linregress(x_adj, y)

    resultados[ticker] = {
        "slope": slope,
        "intercept": intercept,
        "r2": r_value**2,
        "p_value": p_value,
        "std_err": std_err
    }

resultado_df = pd.DataFrame(resultados).T

print(resultado_df.sort_values('slope', ascending=False))

               slope     intercept        r2        p_value   std_err
^BVSP      34.581533  59925.211544  0.847871   0.000000e+00  0.294143
SBSP3.SA    0.034085      5.785399  0.708904   0.000000e+00  0.000439
PRIO3.SA    0.023092     -9.435041  0.878186   0.000000e+00  0.000173
VALE3.SA    0.022857     11.645400  0.742234   0.000000e+00  0.000270
WEGE3.SA    0.019522     -0.148070  0.864383   0.000000e+00  0.000155
RENT3.SA    0.016430     15.240176  0.537006   0.000000e+00  0.000306
SUZB3.SA    0.016218     19.040838  0.705380   0.000000e+00  0.000210
TAEE11.SA   0.013210      4.493849  0.940154   0.000000e+00  0.000067
AXIA6.SA    0.012853      4.912947  0.811366   0.000000e+00  0.000124
PSSA3.SA    0.012769      5.721929  0.726038   0.000000e+00  0.000158
ENGI11.SA   0.012612     14.049713  0.824147   0.000000e+00  0.000117
EQTL3.SA    0.012080      5.288172  0.949374   0.000000e+00  0.000056
CPFE3.SA    0.011514     11.056685  0.861753   0.000000e+00  0.000093
ISAE4.SA    0.009646

## Distribuição de preços - Todos os ativos

Histograma de preços para cada ativo, permitindo visualizar onde o preço atual se posiciona em relação ao histórico.

In [63]:
# Lista de tickers (sem benchmark)
tickers_plot_preco = [t for t in df_core.columns if t != ' ']

# Definir grid: 4 colunas, calcular linhas necessárias
n_cols = 4
n_rows = math.ceil(len(tickers_plot_preco) / n_cols)

# Criar subplots
fig = make_subplots(
    rows=n_rows, 
    cols=n_cols,
    subplot_titles=tickers_plot_preco,
    vertical_spacing=0.06,
    horizontal_spacing=0.04
)

# Adicionar histograma para cada ativo
for i, ticker in enumerate(tickers_plot_preco):
    row = i // n_cols + 1
    col = i % n_cols + 1
    
    # Preço atual (último dia)
    preco_atual = df_core[ticker].iloc[-1]
    
    # Média histórica
    media_preco = df_core[ticker].mean()
    
    # Histograma
    fig.add_trace(
        go.Histogram(
            x=df_core[ticker],
            nbinsx=50,
            marker_color='teal',
            opacity=0.7,
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Linha da média (tênue)
    fig.add_vline(
        x=media_preco,
        line_dash="dot",
        line_color="rgba(128, 128, 128, 0.4)",
        line_width=1,
        row=row, col=col
    )
    
    # Linha vertical do preço atual
    fig.add_vline(
        x=preco_atual,
        line_dash="dash",
        line_color="red",
        line_width=2,
        row=row, col=col
    )
    
    # Anotação do preço atual
    fig.add_annotation(
        x=preco_atual,
        y=1,
        yref=f"y{i+1} domain" if i > 0 else "y domain",
        text=f"R${preco_atual:.2f}",
        showarrow=False,
        font=dict(size=8, color="red"),
        row=row, col=col
    )

# Atualizar layout
fig.update_layout(
    title_text='Distribuição de Preços (R$) - Vermelha = atual | Cinza = média',
    height=250 * n_rows,
    width=1200,
    showlegend=False
)

# Atualizar eixos
fig.update_xaxes(title_text='', tickfont=dict(size=8))
fig.update_yaxes(title_text='', tickfont=dict(size=8))

fig.show()

## Distância dos preços em relação à regressão linear

Mede o quanto o preço atual se afasta da linha de tendência.

**Desvio percentual com sinal**
- **Positivo**: preço atual está acima da linha de tendência (sobrevalorizado).
- **Negativo**: preço atual está abaixo da linha de tendência (subvalorizado).
- Valores próximos de zero indicam que o preço está alinhado com a tendência.

In [64]:
import plotly.express as px

# Calcular distância para cada ativo
distancias = {}

x = np.arange(len(df_core))

for ticker in df_core.columns:
    y = df_core[ticker]
    x_adj = x[-len(y):]
    
    # Regressão linear
    slope, intercept, *_ = linregress(x_adj, y)
    
    # Valores preditos
    y_pred = slope * x_adj + intercept
    
    # Desvio percentual com sinal na última observação
    # Positivo = acima da regressão, Negativo = abaixo da regressão
    desvio_atual = ((y.iloc[-1] - y_pred[-1]) / y_pred[-1]) * 100
    
    distancias[ticker] = desvio_atual

# Criar DataFrame e ordenar
distancia_df = pd.DataFrame(distancias.items(), columns=['Ticker', 'Desvio (%)'])
distancia_df = distancia_df.sort_values('Desvio (%)', ascending=True)

# Plotar gráfico com Plotly
fig = px.bar(
    distancia_df, 
    x='Desvio (%)', 
    y='Ticker',
    orientation='h',
    title='Distância dos Preços em Relação à Regressão Linear<br>(negativo = abaixo, positivo = acima)',
    labels={'Desvio (%)': 'Desvio Percentual (%)'},
    color='Desvio (%)',
    color_continuous_scale='RdBu_r',  # Vermelho = abaixo, Azul = acima
    height=800
)

fig.update_layout(
    xaxis_title='Desvio Percentual (%)',
    yaxis_title='Ticker',
    showlegend=False,
    font=dict(size=11)
)

fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.show()

# Exibir tabelas
# print("Top 10 ativos mais ABAIXO da tendência (subvalorizados):")
# print(distancia_df.head(10).to_string(index=False))
# print("\nTop 10 ativos mais ACIMA da tendência (sobrevalorizados):")
# print(distancia_df.tail(10).to_string(index=False))

In [65]:
# Criar DataFrame separado com distâncias da regressão linear
df_distancia_regressao = pd.DataFrame()

x = np.arange(len(df_core))

for ticker in df_core.columns:
    y = df_core[ticker]
    x_adj = x[-len(y):]
    
    # Regressão linear
    slope, intercept, r_value, p_value, std_err = linregress(x_adj, y)
    
    # Valores preditos pela regressão
    y_pred = slope * x_adj + intercept
    
    # Desvio percentual (preço real - preço previsto) / preço previsto
    desvio_pct = ((y.values - y_pred) / y_pred) * 100
    
    df_distancia_regressao[ticker] = desvio_pct

# Usar o mesmo índice do df_core
df_distancia_regressao.index = df_core.index

df_distancia_regressao

,ABEV3.SA,AXIA6.SA,B3SA3.SA,BBDC3.SA,BBDC4.SA,BRAP4.SA,CPFE3.SA,CPLE3.SA,EGIE3.SA,ENEV3.SA,...,SLCE3.SA,SUZB3.SA,TAEE11.SA,TIMS3.SA,UGPA3.SA,USIM5.SA,VALE3.SA,VIVT3.SA,WEGE3.SA,^BVSP
Date,,,,,,,,,,,,,,,,,,,,,
2016-02-22,-5.429567,-11.075757,-52.547591,-43.285408,-50.164093,1.274440,-0.241834,-264.762865,-4.965579,27.912692,...,74.926925,-11.722596,47.750524,12.255084,36.668612,-87.740444,-44.213271,10.812369,-3199.346962,-27.851736
2016-02-23,-7.104134,-13.795285,-52.915996,-43.712045,-50.646534,-7.537322,-3.483420,-258.002071,-5.764010,27.611403,...,74.261335,-11.797721,49.162954,7.820041,35.934777,-87.473503,-47.542848,8.547640,-3635.810071,-29.084145
2016-02-24,-7.506907,-16.244003,-52.819739,-43.690931,-50.747733,-10.289631,-4.048092,-254.210074,-6.399671,17.518335,...,72.664803,-11.872719,44.566880,8.500899,36.693312,-87.745332,-50.478821,5.550499,-3889.413246,-29.851756
2016-02-25,-9.894033,-15.949429,-53.187490,-43.620085,-50.896570,-18.075888,-1.306372,-259.685041,-5.877734,17.242827,...,72.767112,-11.947590,46.618459,9.519124,36.795603,-87.613137,-53.487742,7.078506,-4761.887530,-30.220342
2016-02-26,-10.296992,-17.528811,-52.796137,-44.096329,-50.759636,-17.678159,-5.637363,-256.887763,-6.512441,16.968607,...,69.806145,-12.022333,48.101489,12.907669,36.609266,-88.019446,-53.662576,7.667093,-5904.335347,-30.751665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-12,51.184722,69.714416,20.127264,45.843142,43.034804,24.197279,27.621679,41.341483,9.594616,14.024245,...,-12.810108,-1.625707,17.884006,68.195834,51.459183,-30.602915,30.716540,51.431685,12.074208,28.974632
2026-02-13,49.636589,68.703035,20.789239,44.244429,41.269938,20.320876,28.165532,41.382770,8.185479,23.177284,...,-12.788190,-1.500690,17.116910,62.198282,50.697744,-27.275120,27.450991,49.418933,11.551947,28.049884
2026-02-18,47.630659,69.187913,21.796723,43.361530,40.852633,17.655795,25.173787,42.435632,6.777080,22.273364,...,-13.095475,-3.451863,17.290190,60.363694,51.667781,-27.284826,22.855431,47.881094,10.657038,27.711902


In [66]:
# Distribuição da distância dos preços em relação à regressão linear - Todos os ativos
tickers_plot_dist = [t for t in df_distancia_regressao.columns if t != ' ']

# Definir grid: 4 colunas, calcular linhas necessárias
n_cols = 4
n_rows = math.ceil(len(tickers_plot_dist) / n_cols)

# Criar subplots
fig = make_subplots(
    rows=n_rows, 
    cols=n_cols,
    subplot_titles=tickers_plot_dist,
    vertical_spacing=0.06,
    horizontal_spacing=0.04
)

# Adicionar histograma para cada ativo
for i, ticker in enumerate(tickers_plot_dist):
    row = i // n_cols + 1
    col = i % n_cols + 1
    
    # Desvio atual (último dia)
    desvio_atual = df_distancia_regressao[ticker].iloc[-1]
    
    # Média histórica do desvio
    media_desvio = df_distancia_regressao[ticker].mean()
    
    # Histograma
    fig.add_trace(
        go.Histogram(
            x=df_distancia_regressao[ticker],
            nbinsx=50,
            marker_color='purple',
            opacity=0.7,
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Linha do zero (referência)
    fig.add_vline(
        x=0,
        line_dash="dot",
        line_color="rgba(128, 128, 128, 0.5)",
        line_width=1,
        row=row, col=col
    )
    
    # Linha vertical do desvio atual
    fig.add_vline(
        x=desvio_atual,
        line_dash="dash",
        line_color="red",
        line_width=2,
        row=row, col=col
    )
    
    # Anotação do desvio atual
    fig.add_annotation(
        x=desvio_atual,
        y=1,
        yref=f"y{i+1} domain" if i > 0 else "y domain",
        text=f"{desvio_atual:.1f}%",
        showarrow=False,
        font=dict(size=8, color="red"),
        row=row, col=col
    )

# Atualizar layout
fig.update_layout(
    title_text='Distribuição da Distância dos Preços da Regressão Linear (%)<br>Vermelha = atual | Cinza = linha de tendência (0%)',
    height=250 * n_rows,
    width=1200,
    showlegend=False
)

# Atualizar eixos
fig.update_xaxes(title_text='', tickfont=dict(size=8))
fig.update_yaxes(title_text='', tickfont=dict(size=8))

fig.show()

In [71]:
# Visualização isolada de um único ativo
ticker_selecionado = 'WEGE3.SA'  # <-- Altere aqui o ticker desejado

# Dados do ativo
serie_desvio = df_distancia_regressao[ticker_selecionado]
desvio_atual = serie_desvio.iloc[-1]
media = serie_desvio.mean()
mediana = serie_desvio.median()
desvio_padrao = serie_desvio.std()

# Criar figura
fig = go.Figure()

# Histograma
fig.add_trace(
    go.Histogram(
        x=serie_desvio,
        nbinsx=50,
        marker_color='purple',
        opacity=0.7,
        name='Distribuição'
    )
)

# Linha do zero (regressão)
fig.add_vline(x=0, line_dash="dot", line_color="gray", line_width=2, 
              annotation_text="Regressão", annotation_position="top")

# Linha da média
fig.add_vline(x=media, line_dash="dash", line_color="blue", line_width=2,
              annotation_text=f"Média: {media:.1f}%", annotation_position="top left")

# Linha do desvio atual
fig.add_vline(x=desvio_atual, line_dash="solid", line_color="red", line_width=3,
              annotation_text=f"Atual: {desvio_atual:.1f}%", annotation_position="top right")

# Bandas de desvio padrão (±1σ e ±2σ)
for n_sigma, opacity in [(1, 0.2), (2, 0.1)]:
    fig.add_vrect(
        x0=media - n_sigma * desvio_padrao,
        x1=media + n_sigma * desvio_padrao,
        fillcolor="blue",
        opacity=opacity,
        line_width=0,
    )

fig.update_layout(
    title=f'Distribuição da Distância do Preço da Regressão Linear - {ticker_selecionado}<br>'
          f'<sup>Média: {media:.2f}% | Mediana: {mediana:.2f}% | σ: {desvio_padrao:.2f}% | Atual: {desvio_atual:.2f}%</sup>',
    xaxis_title='Desvio Percentual (%)',
    yaxis_title='Frequência',
    height=500,
    width=900,
    showlegend=False
)

fig.show()

# Percentil do desvio atual
percentil = (serie_desvio < desvio_atual).sum() / len(serie_desvio) * 100
print(f"O desvio atual ({desvio_atual:.2f}%) está no percentil {percentil:.1f}%")

O desvio atual (7.67%) está no percentil 65.5%


## Distância dos preços em relação à média móvel de 200 períodos

Mede o quanto o preço atual se afasta da média móvel de 200 dias.

**Desvio percentual com sinal**
- **Positivo**: preço atual está acima da MA200 (momentum altista).
- **Negativo**: preço atual está abaixo da MA200 (momentum baixista).
- A MA200 é referência técnica clássica para tendência de longo prazo.

In [68]:
# Calcular distância da MA200 para cada ativo
distancias_ma200 = {}

for ticker in df_core.columns:
    # Calcular média móvel de 200 períodos
    ma200 = df_core[ticker].rolling(window=200).mean()
    
    # Preço atual
    preco_atual = df_core[ticker].iloc[-1]
    
    # MA200 atual
    ma200_atual = ma200.iloc[-1]
    
    # Desvio percentual com sinal
    # Positivo = acima da MA200, Negativo = abaixo da MA200
    desvio = ((preco_atual - ma200_atual) / ma200_atual) * 100
    
    distancias_ma200[ticker] = desvio

# Criar DataFrame e ordenar
distancia_ma200_df = pd.DataFrame(distancias_ma200.items(), columns=['Ticker', 'Desvio MA200 (%)'])
distancia_ma200_df = distancia_ma200_df.sort_values('Desvio MA200 (%)', ascending=True)

# Plotar gráfico com Plotly
fig = px.bar(
    distancia_ma200_df, 
    x='Desvio MA200 (%)', 
    y='Ticker',
    orientation='h',
    title='Distância dos Preços em Relação à MA200<br>(negativo = abaixo, positivo = acima)',
    labels={'Desvio MA200 (%)': 'Desvio Percentual em relação à MA200 (%)'},
    color='Desvio MA200 (%)',
    color_continuous_scale='RdBu_r',  # Vermelho = abaixo, Azul = acima
    height=800
)

fig.update_layout(
    xaxis_title='Desvio Percentual em relação à MA200 (%)',
    yaxis_title='Ticker',
    showlegend=False,
    font=dict(size=11)
)

fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)

fig.show()

# Exibir tabelas
# print("Top 10 ativos mais ABAIXO da MA200 (momentum baixista):")
# print(distancia_ma200_df.head(10).to_string(index=False))
# print("\nTop 10 ativos mais ACIMA da MA200 (momentum altista):")
# print(distancia_ma200_df.tail(10).to_string(index=False))

# **Retornos**

## Distribuição de retornos diários - Todos os ativos

Histograma de retornos diários para cada ativo, permitindo visualizar a forma da distribuição (normalidade, assimetria, caudas).

In [69]:
# Lista de tickers (sem benchmark)
tickers_plot = [t for t in df_ret.columns if t != '^BVSP']

# Definir grid: 4 colunas, calcular linhas necessárias
n_cols = 4
n_rows = math.ceil(len(tickers_plot) / n_cols)

# Criar subplots
fig = make_subplots(
    rows=n_rows, 
    cols=n_cols,
    subplot_titles=tickers_plot,
    vertical_spacing=0.06,
    horizontal_spacing=0.04
)

# Adicionar histograma para cada ativo
for i, ticker in enumerate(tickers_plot):
    row = i // n_cols + 1
    col = i % n_cols + 1
    
    # Retorno atual (último dia)
    retorno_atual = df_ret[ticker].iloc[-1] * 100
    
    # Histograma
    fig.add_trace(
        go.Histogram(
            x=df_ret[ticker] * 100,  # Converter para percentual
            nbinsx=50,
            marker_color='steelblue',
            opacity=0.7,
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Linha vertical do retorno atual
    fig.add_vline(
        x=retorno_atual,
        line_dash="dash",
        line_color="red",
        line_width=2,
        row=row, col=col
    )
    
    # Anotação do retorno atual
    fig.add_annotation(
        x=retorno_atual,
        y=1,
        yref=f"y{i+1} domain" if i > 0 else "y domain",
        text=f"{retorno_atual:.2f}%",
        showarrow=False,
        font=dict(size=8, color="red"),
        row=row, col=col
    )

# Atualizar layout
fig.update_layout(
    title_text='Distribuição de Retornos Diários (%) - Linha vermelha = retorno atual',
    height=250 * n_rows,
    width=1200,
    showlegend=False
)

# Atualizar eixos
fig.update_xaxes(title_text='', tickfont=dict(size=8))
fig.update_yaxes(title_text='', tickfont=dict(size=8))

fig.show()

## Estatísticas de regressão
Modelo CAPM: $R_i - R_f = \alpha + \beta(R_m - R_f) + \varepsilon$

Simplificado (sem taxa livre de risco): $R_i = \alpha + \beta \cdot R_m + \varepsilon$

**Métricas calculadas:**
- **Beta (β)**: sensibilidade do ativo ao benchmark.
- **Alpha (α)**: retorno adicional não explicado pelo benchmark.
- **R²**: percentual da variância explicada pelo modelo (qualidade do ajuste).
- **t-stat**: razão entre beta e seu erro padrão (significância estatística).
- **p-value**: probabilidade do beta ser zero (relevância do benchmark).

In [70]:
# DataFrame para armazenar todas as métricas
regression_stats = {}

# Definir benchmark (IBOVESPA)
benchmark = df_ret['^BVSP']

for ticker in df_ret.columns:
    if ticker == '^BVSP':
        continue
    
    # Preparar dados
    y = df_ret[ticker]
    x = benchmark.loc[y.index]
    
    # Remover NaN comuns
    mask = ~(x.isna() | y.isna())
    x_clean = x[mask]
    y_clean = y[mask]
    
    # Número de observações
    n = len(x_clean)
    
    # Regressão linear
    slope, intercept, r_value, p_value, std_err = linregress(x_clean, y_clean)
    
    # Beta
    beta = slope
    
    # Alpha (anualizado, assumindo retornos diários)
    alpha = intercept * 252
    
    # R²
    r_squared = r_value ** 2
    
    # t-statistic para beta
    t_stat = beta / std_err if std_err != 0 else np.nan
    
    # p-value já vem do linregress (testa H0: beta = 0)
    p_val = p_value
    
    regression_stats[ticker] = {
        'Beta': beta,
        'Alpha (anual)': alpha,
        'R²': r_squared,
        't-stat': t_stat,
        'p-value': p_val,
        'n_obs': n
    }

# Criar DataFrame
regression_df = pd.DataFrame(regression_stats).T

# Formatar para exibição
regression_df_display = regression_df.copy()
regression_df_display['Beta'] = regression_df_display['Beta'].map('{:.4f}'.format)
regression_df_display['Alpha (anual)'] = regression_df_display['Alpha (anual)'].map('{:.2%}'.format)
regression_df_display['R²'] = regression_df_display['R²'].map('{:.4f}'.format)
regression_df_display['t-stat'] = regression_df_display['t-stat'].map('{:.2f}'.format)
regression_df_display['p-value'] = regression_df_display['p-value'].map(lambda x: f'{x:.4f}' if x >= 0.0001 else '<0.0001')
regression_df_display.sort_values('Beta', ascending=False)

,Beta,Alpha (anual),R²,t-stat,p-value,n_obs
USIM5.SA,1.4159,11.49%,0.3629,37.58,<0.0001,2481.0
GOAU4.SA,1.2643,15.98%,0.4343,43.62,<0.0001,2481.0
RENT3.SA,1.2000,10.16%,0.4394,44.08,<0.0001,2481.0
B3SA3.SA,1.1808,5.58%,0.5319,53.07,<0.0001,2481.0
PRIO3.SA,1.1802,56.00%,0.1844,23.67,<0.0001,2481.0
GGBR4.SA,1.1791,11.85%,0.4145,41.89,<0.0001,2481.0
MRVE3.SA,1.1750,-5.24%,0.3356,35.39,<0.0001,2481.0
BBDC4.SA,1.1697,-1.17%,0.6141,62.81,<0.0001,2481.0
UGPA3.SA,1.1514,1.61%,0.2356,27.64,<0.0001,2481.0
BBDC3.SA,1.1375,-3.42%,0.6404,66.44,<0.0001,2481.0


## Interpretação das métricas

**Beta (β)**
- `β > 1`: ativo mais volátil que o benchmark.
- `β = 1`: ativo acompanha o benchmark.
- `β < 1`: ativo menos volátil (defensivo).

**Alpha (α)**
- Retorno anualizado não explicado pelo benchmark.
- `α > 0`: desempenho superior ao esperado pelo modelo.
- `α < 0`: desempenho inferior ao esperado.

**R²**
- Percentual da variação do ativo explicado pelo benchmark.
- `R² = 0,85`: 85% da variação é explicada pelo mercado.
- `R² baixo`: ativo pouco correlacionado ao benchmark.

**t-stat**
- Razão entre o beta e seu erro-padrão.
- `|t| > 2`: beta estatisticamente relevante (aprox. 95% de confiança).
- `|t| < 2`: beta pode não ser significativo.

**p-value**
- Significância estatística do beta.
- `p < 0,05`: forte evidência de que o beta é diferente de zero.
- `p >= 0,05`: beta pode ser zero (sem relação com benchmark).